# Phase 4 — Paired boundary evaluation

Use the existing Kaggle session. If it reset, attach BOTH phase3_results.zip and
phase4_results.zip as private inputs. ZIP and automatically extracted inputs are
supported and verified. Enable Internet for raw dataset restoration if needed.
No training occurs. Both validation replays must pass before held-out evaluation.


In [11]:
files={'src/atomic.py': '"""Durable replacement: failed writes never replace an existing result."""\nimport json\nimport os\nfrom pathlib import Path\nimport tempfile\n\n\ndef atomic_write(path, writer):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")\n    try:\n        with os.fdopen(fd, "wb") as stream:\n            writer(stream)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef write_json(path, value):\n    atomic_write(path, lambda stream: stream.write(\n        (json.dumps(value, indent=2, sort_keys=True) + "\\n").encode("utf-8")))\n', 'src/boundaries.py': "import torch\nfrom torch.nn import functional as F\n\ndef boundary_target(mask):\n    # Replicated borders avoid inventing edges at the image perimeter.\n    padded=F.pad(mask,(1,1,1,1),mode='replicate')\n    return (F.max_pool2d(padded,3,1)+F.max_pool2d(-padded,3,1)).clamp(0,1)\n\ndef boundary_loss(logits,mask):\n    target=boundary_target(mask)\n    # Fixed weight avoids unstable image-specific weighting on empty masks.\n    return F.binary_cross_entropy_with_logits(logits,target,pos_weight=logits.new_tensor(5.))\n\ndef boundary_f1(logits,mask):\n    pred=boundary_target((logits>=0).float())>0\n    truth=boundary_target(mask)>0\n    # Symmetric many-to-one matching within a two-pixel Chebyshev radius.\n    near_truth=F.max_pool2d(truth.float(),5,1,2)>0\n    near_pred=F.max_pool2d(pred.float(),5,1,2)>0\n    dims=(1,2,3)\n    pcount=pred.sum(dims); tcount=truth.sum(dims)\n    precision=(pred&near_truth).sum(dims)/pcount.clamp_min(1)\n    recall=(truth&near_pred).sum(dims)/tcount.clamp_min(1)\n    score=2*precision*recall/(precision+recall).clamp_min(1e-8)\n    return torch.where((pcount==0)&(tcount==0),1.,score)\n", 'src/checkpoints.py': '"""Trusted-local PyTorch recovery snapshots with atomic pointer updates.\n\nCaller must checkpoint only after a complete optimizer step, and provide sampler\norder/cursor. Exact mid-epoch replay requires num_workers=0 and no prefetched\naugmentation. External backups remain the caller\'s responsibility.\n"""\nimport hashlib\nimport random\nfrom pathlib import Path\nimport uuid\n\nfrom src.atomic import atomic_write, write_json\n\n\ndef save_checkpoint(run_dir, *, model, optimizer, scheduler, scaler,\n                    epoch, global_step, sampler_state, config, manifest_hash,\n                    metrics, best_metric, is_best=False):\n    import numpy as np\n    import torch\n    root = Path(run_dir)\n    root.mkdir(parents=True, exist_ok=True)\n    state = {\n        "schema_version": 1, "model": model.state_dict(),\n        "optimizer": optimizer.state_dict(),\n        "scheduler": scheduler.state_dict() if scheduler is not None else None,\n        "scaler": scaler.state_dict() if scaler is not None else None,\n        "epoch": epoch, "global_step": global_step,\n        "sampler_state": sampler_state, "config": config,\n        "manifest_hash": manifest_hash, "metrics": metrics,\n        "best_metric": best_metric,\n        "rng": {"python": random.getstate(), "numpy": np.random.get_state(),\n                "torch": torch.get_rng_state(),\n                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},\n    }\n    # Unique immutable files preserve older results even if pointer updates fail.\n    name = f"step-{global_step:09d}-{uuid.uuid4().hex[:8]}.pt"\n    target = root / name\n    atomic_write(target, lambda stream: torch.save(state, stream))\n    pointer = {"file": name, "sha256": hashlib.sha256(target.read_bytes()).hexdigest(),\n               "epoch": epoch, "global_step": global_step}\n    write_json(root / "latest.json", pointer)\n    if is_best:\n        write_json(root / "best.json", pointer)\n    return target\n\n\ndef load_checkpoint(run_dir, *, model, optimizer, scheduler, scaler,\n                    expected_manifest_hash, expected_config, which="latest"):\n    import json\n    import numpy as np\n    import torch\n    if which not in {"latest", "best"}:\n        raise ValueError("which must be latest or best")\n    root = Path(run_dir).resolve()\n    pointer = json.loads((root / f"{which}.json").read_text())\n    target = (root / pointer["file"]).resolve()\n    if target.parent != root:\n        raise ValueError("Checkpoint path escapes run directory")\n    if hashlib.sha256(target.read_bytes()).hexdigest() != pointer["sha256"]:\n        raise ValueError("Checkpoint checksum mismatch")\n    # Full optimizer/RNG recovery requires pickle: load only your trusted snapshots.\n    state = torch.load(target, map_location="cpu", weights_only=False)\n    if state["schema_version"] != 1 or state["manifest_hash"] != expected_manifest_hash:\n        raise ValueError("Checkpoint schema or dataset manifest mismatch")\n    if state["config"] != expected_config:\n        raise ValueError("Configuration changed; create a new run instead of resuming")\n    for key, component in [("scheduler", scheduler), ("scaler", scaler)]:\n        if (state[key] is None) != (component is None):\n            raise ValueError(f"Recovery component mismatch: {key}")\n    model.load_state_dict(state["model"])\n    optimizer.load_state_dict(state["optimizer"])\n    if scheduler is not None:\n        scheduler.load_state_dict(state["scheduler"])\n    if scaler is not None:\n        scaler.load_state_dict(state["scaler"])\n    random.setstate(state["rng"]["python"])\n    np.random.set_state(state["rng"]["numpy"])\n    torch.set_rng_state(state["rng"]["torch"])\n    if state["rng"]["cuda"] is not None:\n        if not torch.cuda.is_available():\n            raise ValueError("Exact CUDA recovery requires a compatible CUDA environment")\n        torch.cuda.set_rng_state_all(state["rng"]["cuda"])\n    return state\n', 'src/dataset.py': '"""Manifest-only data access; query records cannot enter the training split."""\nimport hashlib\nfrom pathlib import Path\nimport random\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\n\n\ndef verify_files(records,root):\n    root=Path(root).resolve()\n    for row in records:\n        for field in (\'image\',\'mask\'):\n            path=(root/row[field]).resolve()\n            if not path.is_relative_to(root): raise ValueError(\'Dataset path escapes root\')\n            if hashlib.sha256(path.read_bytes()).hexdigest()!=row[field+\'_sha256\']:\n                raise ValueError(f"Dataset checksum changed: {row[\'image_id\']} {field}")\n\n\nclass CrackDataset:\n    def __init__(self,manifest,root,split,size=256,augment=False):\n        if augment and split!=\'source_train\': raise ValueError(\'Only source_train may augment\')\n        self.rows=[r for r in manifest[\'records\'] if r[\'split\']==split]\n        if not self.rows: raise ValueError(\'Empty split: \'+split)\n        self.root=Path(root); self.size=size; self.augment=augment\n    def __len__(self): return len(self.rows)\n    def __getitem__(self,index):\n        row=self.rows[index]\n        with Image.open(self.root/row[\'image\']) as source: image=source.convert(\'RGB\')\n        with Image.open(self.root/row[\'mask\']) as source:\n            raw=np.asarray(source)\n            if raw.ndim==3:\n                if not np.all(raw[...,0]==raw[...,1]) or not np.all(raw[...,0]==raw[...,2]):\n                    raise ValueError(\'Colored mask requires explicit mapping\')\n                raw=raw[...,0]\n            if not set(np.unique(raw).tolist())<={0,255}: raise ValueError(\'Nonbinary mask\')\n            mask=Image.fromarray(raw.astype(np.uint8))\n        image=image.resize((self.size,self.size),Image.Resampling.BILINEAR)\n        mask=mask.resize((self.size,self.size),Image.Resampling.NEAREST)\n        if self.augment:\n            if random.random()<.5: image,mask=ImageOps.mirror(image),ImageOps.mirror(mask)\n            if random.random()<.5: image,mask=ImageOps.flip(image),ImageOps.flip(mask)\n        return (torch.from_numpy(np.array(image,dtype=np.float32).transpose(2,0,1)/255),\n                torch.from_numpy((np.array(mask)>0).astype(np.float32)[None]))\n', 'src/litecrack.py': '"""Region-only compact model; no boundary head or pretrained external data."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef replace_batchnorm(module):\n    for name,child in list(module.named_children()):\n        if isinstance(child,nn.BatchNorm2d):\n            setattr(module,name,nn.GroupNorm(1,child.num_features))\n        else: replace_batchnorm(child)\n\n\nclass SeparableBlock(nn.Module):\n    def __init__(self,incoming,outgoing):\n        super().__init__()\n        self.layers=nn.Sequential(\n            nn.Conv2d(incoming,incoming,3,padding=1,groups=incoming,bias=False),\n            nn.Conv2d(incoming,outgoing,1,bias=False),nn.GroupNorm(1,outgoing),nn.ReLU(),\n            nn.Conv2d(outgoing,outgoing,3,padding=1,groups=outgoing,bias=False),\n            nn.Conv2d(outgoing,outgoing,1,bias=False),nn.GroupNorm(1,outgoing),nn.ReLU())\n    def forward(self,x): return self.layers(x)\n\n\nclass LiteCrack(nn.Module):\n    def __init__(self,width_mult=.5,boundary_aware=False):\n        super().__init__()\n        from torchvision.models import mobilenet_v3_small\n        # weights=None prevents both downloads and unreported ImageNet exposure.\n        self.encoder=mobilenet_v3_small(weights=None,width_mult=width_mult).features\n        replace_batchnorm(self.encoder)\n        self.indices=(0,1,3,8,12)\n        channels=[self.encoder[i].out_channels for i in self.indices]\n        decoder_widths=(128,64,32,16)\n        current=channels[-1]\n        self.decoder=nn.ModuleList()\n        for skip,width in zip(reversed(channels[:-1]),decoder_widths):\n            self.decoder.append(SeparableBlock(current+skip,width)); current=width\n        self.refine=SeparableBlock(current,current)\n        self.output=nn.Conv2d(current,1,1)\n        self.boundary=None\n        if boundary_aware:\n            # Keep the region initialization and subsequent sampler RNG identical\n            # to the region-only control for the same seed.\n            with torch.random.fork_rng(devices=[]):\n                self.boundary=nn.Conv2d(current,1,1)\n    def forward(self,x):\n        original_size=x.shape[-2:]\n        skips=[]\n        for index,layer in enumerate(self.encoder):\n            x=layer(x)\n            if index in self.indices: skips.append(x)\n        for block,skip in zip(self.decoder,reversed(skips[:-1])):\n            x=F.interpolate(x,size=skip.shape[-2:],mode=\'nearest\')\n            x=block(torch.cat([x,skip],dim=1))\n        x=F.interpolate(x,size=original_size,mode=\'nearest\')\n        features=self.refine(x)\n        region=self.output(features)\n        return (region,self.boundary(features)) if self.boundary is not None else region\n', 'src/models.py': '"""U-Net and a clearly labeled paper-inspired recurrent/attention baseline."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef normalization_layer(channels,normalization):\n    return nn.GroupNorm(1,channels) if normalization==\'group\' else nn.Identity()\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, incoming, outgoing,normalization=\'none\'):\n        super().__init__()\n        self.layers=nn.Sequential(nn.Conv2d(incoming,outgoing,3,padding=1),normalization_layer(outgoing,normalization),nn.ReLU(),\n                                  nn.Conv2d(outgoing,outgoing,3,padding=1),normalization_layer(outgoing,normalization),nn.ReLU())\n    def forward(self,x): return self.layers(x)\n\n\nclass RecurrentResidual(nn.Module):\n    def __init__(self,incoming,outgoing,normalization=\'none\'):\n        super().__init__()\n        self.project=nn.Conv2d(incoming,outgoing,1)\n        self.units=nn.ModuleList([nn.Conv2d(outgoing,outgoing,3,padding=1) for _ in range(2)])\n        self.norms=nn.ModuleList([normalization_layer(outgoing,normalization) for _ in range(2)])\n    def forward(self,x):\n        residual=self.project(x)\n        h=residual\n        for unit,norm in zip(self.units,self.norms):\n            fixed=h\n            h=F.relu(norm(unit(fixed)))\n            for _ in range(2): h=F.relu(norm(unit(fixed+h)))\n        return residual+h\n\n\nclass AttentionGate(nn.Module):\n    def __init__(self,skip_channels,gate_channels):\n        super().__init__()\n        hidden=max(1,skip_channels//2)\n        self.skip=nn.Conv2d(skip_channels,hidden,1)\n        self.gate=nn.Conv2d(gate_channels,hidden,1)\n        self.score=nn.Conv2d(hidden,1,1)\n    def forward(self,skip,gate):\n        return skip*torch.sigmoid(self.score(F.relu(self.skip(skip)+self.gate(gate))))\n\n\nclass SegmentationUNet(nn.Module):\n    def __init__(self,architecture=\'unet\',base_channels=16,normalization=\'none\'):\n        super().__init__()\n        if architecture not in (\'unet\',\'r2au\'): raise ValueError(\'Unknown architecture\')\n        if normalization not in (\'none\',\'group\'): raise ValueError(\'Unknown normalization\')\n        block=ConvBlock if architecture==\'unet\' else RecurrentResidual\n        widths=[base_channels*2**i for i in range(5)]\n        self.encoders=nn.ModuleList([block(3 if i==0 else widths[i-1],w,normalization) for i,w in enumerate(widths)])\n        self.decoders=nn.ModuleList([block(widths[i+1]+widths[i],widths[i],normalization) for i in range(3,-1,-1)])\n        self.attention=nn.ModuleList([AttentionGate(widths[i],widths[i+1]) for i in range(3,-1,-1)]) if architecture==\'r2au\' else None\n        self.output=nn.Conv2d(widths[0],1,1)\n    def forward(self,x):\n        skips=[]\n        for i,encoder in enumerate(self.encoders):\n            if i: x=F.max_pool2d(x,2)\n            x=encoder(x); skips.append(x)\n        for i,decoder in enumerate(self.decoders):\n            skip=skips[-2-i]\n            # Nearest resizing avoids CUDA bilinear-backward nondeterminism in\n            # the strict recovery baseline; compact-model experiments are separate.\n            x=F.interpolate(x,size=skip.shape[-2:],mode=\'nearest\')\n            if self.attention is not None: skip=self.attention[i](skip,x)\n            x=decoder(torch.cat([x,skip],dim=1))\n        return self.output(x)\n', 'src/model_factory.py': '"""Explicit model selection shared by training and profiling."""\nfrom src.models import SegmentationUNet\n\n\ndef build_model(config):\n    if config[\'architecture\'] in (\'litecrack\',\'litecrack_ba\'):\n        from src.litecrack import LiteCrack\n        return LiteCrack(config.get(\'encoder_width_mult\',.5),config[\'architecture\']==\'litecrack_ba\')\n    return SegmentationUNet(config[\'architecture\'],config[\'base_channels\'],config.get(\'normalization\',\'none\'))\n', 'src/segmentation.py': '"""Binary loss/metrics with explicit per-image averaging and empty-mask rules."""\nimport torch\nfrom torch.nn import functional as F\n\n\ndef dice_loss(logits,target):\n    probability=logits.sigmoid()\n    intersection=(probability*target).sum((1,2,3))\n    denominator=(probability+target).sum((1,2,3))\n    return (1-(2*intersection+1e-6)/(denominator+1e-6)).mean()\n\n\ndef region_loss(logits,target,kind=\'dice\'):\n    if kind==\'dice\': return dice_loss(logits,target)\n    if kind==\'bce_dice\': return F.binary_cross_entropy_with_logits(logits,target)+dice_loss(logits,target)\n    raise ValueError(\'Unknown region loss\')\n\n\ndef image_metrics(logits,target):\n    predicted=logits.sigmoid()>=.5\n    truth=target>=.5\n    dimensions=(1,2,3)\n    tp=(predicted & truth).sum(dimensions).float()\n    fp=(predicted & ~truth).sum(dimensions).float()\n    fn=(~predicted & truth).sum(dimensions).float()\n    dice_den=2*tp+fp+fn\n    union=tp+fp+fn\n    return {\'dice\':torch.where(dice_den==0,1,2*tp/dice_den.clamp_min(1)),\n            \'iou\':torch.where(union==0,1,tp/union.clamp_min(1)),\n            \'precision\':torch.where(tp+fp==0,(fn==0).float(),tp/(tp+fp).clamp_min(1)),\n            \'recall\':torch.where(tp+fn==0,1,tp/(tp+fn).clamp_min(1))}\n', 'src/__init__.py': '"""Road crack segmentation research package."""\n', 'scripts/audit_kaggle_data.py': '"""Inventory real Kaggle data and produce reviewable candidate splits.\n\nNever invent sequence metadata: similarity clusters are only proxy groups.\nNo manifest is frozen until annotation polarity and independence are reviewed.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport re\nimport sys\nimport zipfile\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.atomic import write_json\n\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.tif\', \'.tiff\'}\n\n\ndef filename_group(dataset, key):\n    # Group numbered patches sharing a GOPR identifier conservatively.\n    if dataset == \'CrackMap\':\n        match = re.fullmatch(r\'(GOPR\\d+)(?: \\(\\d+\\))?\', key)\n        if match:\n            return \'filename_capture_\' + match.group(1)\n    return None\n\n\ndef source_layout(data_root):\n    directory = Path(data_root)/\'DeepCrack/dataset\'\n    archive = directory/\'DeepCrack.zip\'\n    def locate():\n        return [p for p in directory.rglob(\'train_img\') if p.is_dir()\n                and all((p.parent/name).is_dir() for name in (\'train_lab\',\'test_img\',\'test_lab\'))]\n    candidates = locate() if directory.exists() else []\n    if not candidates and archive.exists():\n        destination = directory/\'extracted_v1\'\n        destination.mkdir(exist_ok=True)\n        with zipfile.ZipFile(archive) as z:\n            for entry in z.infolist():\n                target = (destination/entry.filename).resolve()\n                if not target.is_relative_to(destination.resolve()):\n                    raise ValueError(\'Unsafe DeepCrack ZIP path\')\n                if ((entry.external_attr >> 16) & 0o170000) == 0o120000:\n                    raise ValueError(\'Symlink in DeepCrack ZIP\')\n                if not entry.is_dir() and target.exists() and target.read_bytes()!=z.read(entry):\n                    raise ValueError(\'Existing extracted dataset differs; preserve and inspect\')\n            z.extractall(destination)\n        candidates = locate()\n    if len(candidates)!=1:\n        raise ValueError(f\'Expected one complete DeepCrack layout, found {len(candidates)}\')\n    return candidates[0].parent\n\n\ndef file_map(directory):\n    if not directory.is_dir():\n        raise ValueError(f\'Missing dataset directory: {directory}\')\n    result = {}\n    for path in sorted(directory.rglob(\'*\')):\n        if path.is_file() and path.suffix.lower() in EXTENSIONS:\n            # Only remove documented annotation suffixes; report all ambiguity.\n            key = path.stem\n            for suffix in (\'_mask\', \'_label\', \'_lab\'):\n                if key.endswith(suffix):\n                    key = key[:-len(suffix)]\n                    break\n            if key in result:\n                raise ValueError(f\'Ambiguous pair key {key}: {result[key]} / {path}\')\n            result[key] = path\n    return result\n\n\ndef paired_files(images, masks):\n    left, right = file_map(images), file_map(masks)\n    if not left or set(left) != set(right):\n        raise ValueError(f\'Unpaired/empty folders: {images}; \'\n                         f\'images_without_masks={sorted(set(left)-set(right))[:20]}, \'\n                         f\'masks_without_images={sorted(set(right)-set(left))[:20]}\')\n    return [(key, left[key], right[key]) for key in sorted(left)]\n\n\ndef audit(data_root, output, seed=42, threshold=None):\n    import numpy as np\n    from PIL import Image, ImageDraw\n    output = Path(output).resolve()\n    if output.exists():\n        raise ValueError(\'Refusing to overwrite audit evidence; choose a new output directory\')\n    output.mkdir(parents=True)\n    layouts = [\n        (\'DeepCrack\', \'source_train\', \'DeepCrack/dataset/train_img\', \'DeepCrack/dataset/train_lab\'),\n        (\'DeepCrack\', \'source_test\', \'DeepCrack/dataset/test_img\', \'DeepCrack/dataset/test_lab\'),\n        (\'CrackMap\', \'target\', \'CrackMap/images\', \'CrackMap/masks\'),\n    ]\n    records, errors, conversions = [], [], []\n    try:\n        source = source_layout(data_root)\n        layouts[:2] = [\n            (\'DeepCrack\',\'source_train\',source/\'train_img\',source/\'train_lab\'),\n            (\'DeepCrack\',\'source_test\',source/\'test_img\',source/\'test_lab\')]\n    except Exception as error:\n        errors.append(str(error))\n    derived = output / \'derived_masks\'\n    for dataset, partition, image_dir, mask_dir in layouts:\n        try:\n            pairs = paired_files(Path(data_root)/image_dir, Path(data_root)/mask_dir)\n        except Exception as error:\n            errors.append(str(error))\n            continue\n        for key, image_path, mask_path in pairs:\n            try:\n                with Image.open(image_path) as source:\n                    image = source.convert(\'RGB\')\n                    image.load()\n                with Image.open(mask_path) as source:\n                    raw = np.asarray(source)\n                if raw.ndim == 3:\n                    if raw.shape[2] not in (3, 4) or not np.all(raw[..., 0] == raw[..., 1]) or not np.all(raw[..., 0] == raw[..., 2]):\n                        raise ValueError(\'Colored annotations need an explicit label mapping\')\n                    if raw.shape[2] == 4 and not np.all(raw[..., 3] == 255):\n                        raise ValueError(\'Transparent annotations need an explicit label mapping\')\n                    raw = raw[..., 0]\n                if raw.ndim != 2 or image.size != (raw.shape[1], raw.shape[0]):\n                    raise ValueError(\'Mask dimensions do not match image\')\n                values = np.unique(raw)\n                if set(values.tolist()) <= {0, 1, 255}:\n                    binary = (raw > 0).astype(np.uint8)*255\n                    rule = \'positive label values become 255; zero becomes 0\'\n                elif threshold is not None:\n                    binary = (raw >= threshold).astype(np.uint8)*255\n                    rule = f\'explicit mask threshold >= {threshold}\'\n                else:\n                    raise ValueError(f\'Nonbinary mask ({len(values)} values); review before explicit thresholding\')\n                destination = derived/dataset/partition/f\'{key}.png\'\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                Image.fromarray(binary).save(destination)\n                small = np.asarray(image.convert(\'L\').resize((9,8), Image.Resampling.LANCZOS))\n                dhash = int(\'\'.join(\'1\' if b else \'0\' for b in (small[:,1:] > small[:,:-1]).flat),2)\n                records.append({\'image_id\': dataset+\':\'+key, \'dataset\':dataset, \'partition\':partition,\n                    \'image\':str(image_path.resolve()), \'mask\':str(destination),\n                    \'raw_mask\':str(mask_path.resolve()), \'mask_values\':values.tolist(),\n                    \'image_sha256\':hashlib.sha256(image_path.read_bytes()).hexdigest(),\n                    \'raw_mask_sha256\':hashlib.sha256(mask_path.read_bytes()).hexdigest(),\n                    \'mask_sha256\':hashlib.sha256(destination.read_bytes()).hexdigest(),\n                    \'pixel_sha256\':hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest(),\n                    \'dhash\':f\'{dhash:016x}\', \'width\':image.width, \'height\':image.height,\n                    \'foreground_fraction\':float((binary>0).mean())})\n                conversions.append({\'image_id\':dataset+\':\'+key,\'rule\':rule})\n            except Exception as error:\n                errors.append(f\'{dataset}/{partition}/{key}: {error}\')\n\n    # Similarity components keep detected related images together. They do not\n    # establish road/sequence independence and require human review.\n    parent = list(range(len(records)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n    matches = []\n    known_groups = {}\n    for i, row in enumerate(records):\n        group = filename_group(row[\'dataset\'], row[\'image_id\'].split(\':\',1)[1])\n        if group:\n            if group in known_groups:\n                parent[find(i)] = find(known_groups[group])\n            known_groups[group] = i\n        row[\'filename_group\'] = group\n    for i, row in enumerate(records):\n        for j in range(i):\n            other = records[j]\n            exact = row[\'pixel_sha256\'] == other[\'pixel_sha256\']\n            distance = (int(row[\'dhash\'],16)^int(other[\'dhash\'],16)).bit_count()\n            if exact or distance <= 3:\n                parent[find(i)] = find(j)\n                matches.append({\'a\':row[\'image_id\'],\'b\':other[\'image_id\'],\n                                \'exact\':exact,\'dhash_distance\':distance})\n                if row[\'partition\'] != other[\'partition\']:\n                    errors.append(f"Similarity crosses native partitions: {row[\'image_id\']} / {other[\'image_id\']}")\n    for i,row in enumerate(records):\n        row[\'group_id\'] = \'similarity_proxy_\' + str(find(i))\n        row[\'group_provenance\'] = \'GOPR filename capture + decoded hash + dHash<=3; site/sequence independence NOT verified\'\n    rng = random.Random(seed)\n    allocations = {}\n    for partition, selected, remaining, fraction in [\n        (\'source_train\',\'source_validation\',\'source_train\',.2),\n        (\'target\',\'target_acquisition\',\'target_query\',.25)]:\n        groups = sorted({r[\'group_id\'] for r in records if r[\'partition\']==partition})\n        if len(groups)<2:\n            errors.append(f\'Not enough independent candidate groups for {partition}\')\n            continue\n        rng.shuffle(groups)\n        n = max(1,min(len(groups)-1,round(len(groups)*fraction)))\n        allocations.update({(partition,g):selected if i<n else remaining for i,g in enumerate(groups)})\n    for row in records:\n        row[\'split\'] = (\'source_holdout\' if row[\'partition\']==\'source_test\' else\n                        allocations.get((row[\'partition\'],row[\'group_id\']),\'unassigned\'))\n    counts = {s:sum(r[\'split\']==s for r in records) for s in sorted({r[\'split\'] for r in records})}\n    manifest = {\'schema_version\':1,\'status\':\'CANDIDATE_NOT_FROZEN\',\'seed\':seed,\n                \'counts\':counts,\'records\':records,\'errors\':errors,\n                \'limitations\':[\'White=crack polarity awaits contact-sheet review.\',\n                               \'GOPR filename captures stay together; other similarity groups are proxies, not verified parent frames.\',\n                               \'Near-duplicate detection is heuristic and may miss related captures.\']}\n    write_json(output/\'candidate_manifest.json\',manifest)\n    write_json(output/\'similarity_matches.json\',matches)\n    write_json(output/\'mask_conversions.json\',conversions)\n    if records:\n        with (output/\'candidate_index.csv\').open(\'w\',newline=\'\') as stream:\n            writer=csv.DictWriter(stream,fieldnames=[\'image_id\',\'image\',\'mask\',\'dataset\',\'group_id\',\'partition\'])\n            writer.writeheader()\n            writer.writerows({key:r[key] for key in writer.fieldnames} for r in records)\n        # Up to eight examples per native partition, sampled deterministically.\n        chosen=[]\n        for partition in (\'source_train\',\'source_test\',\'target\'):\n            pool=[r for r in records if r[\'partition\']==partition]\n            chosen.extend(random.Random(seed).sample(pool,min(8,len(pool))))\n        sheet=Image.new(\'RGB\',(768, len(chosen)*155),\'white\')\n        draw=ImageDraw.Draw(sheet)\n        for i,row in enumerate(chosen):\n            image=Image.open(row[\'image\']).convert(\'RGB\').resize((256,128))\n            mask=Image.open(row[\'mask\']).convert(\'L\').resize((256,128),Image.Resampling.NEAREST)\n            overlay=Image.blend(image,Image.new(\'RGB\',image.size,\'red\'),.55)\n            overlay=Image.composite(overlay,image,mask)\n            for x,view in enumerate((image,mask.convert(\'RGB\'),overlay)):\n                sheet.paste(view,(x*256,i*155+22))\n            draw.text((2,i*155+3),row[\'image_id\']+\' | original / mask / overlay\',fill=\'black\')\n        sheet.save(output/\'contact_sheet.jpg\',quality=90)\n    # This inventory documents the real folders even if pairing failed.\n    write_json(output/\'inventory.json\',[\n        str(p.relative_to(data_root)) for p in sorted(Path(data_root).rglob(\'*\'))\n        if p.is_file() and \'.git\' not in p.parts])\n    report={\'status\':\'review_required\',\'valid_pairs\':len(records),\'counts\':counts,\n            \'error_count\':len(errors),\'errors\':errors,\'similarity_matches\':len(matches),\n            \'frozen\':False,\'training_started\':False}\n    write_json(output/\'audit_report.json\',report)\n    print(json.dumps(report,indent=2))\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data-root\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--seed\',type=int,default=42)\n    parser.add_argument(\'--mask-threshold\',type=int)\n    args=parser.parse_args()\n    if args.mask_threshold is not None and not 1<=args.mask_threshold<=255:\n        parser.error(\'mask threshold must be between 1 and 255\')\n    audit(args.data_root,args.output,args.seed,args.mask_threshold)\n', 'scripts/compare_boundaries.py': '"""Paired evaluation of frozen region-only and boundary-aware checkpoints."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport torch\nfrom src.atomic import write_json\nfrom src.dataset import CrackDataset,verify_files\nfrom src.model_factory import build_model\nfrom scripts.train_baseline import evaluate\n\n\ndef compare(control,treatment,data_root,output):\n    control,treatment,output=Path(control),Path(treatment),Path(output)\n    if output.exists(): raise ValueError(\'Preserve prior evaluation; choose a new output directory\')\n    manifest_bytes=(treatment/\'data/manifests/pilot_v1.json\').read_bytes()\n    if manifest_bytes!=(control/\'data/manifests/pilot_v1.json\').read_bytes(): raise ValueError(\'Split mismatch\')\n    manifest_hash=hashlib.sha256(manifest_bytes).hexdigest()\n    if manifest_hash!=(treatment/\'data/manifests/pilot_v1.json.sha256\').read_text().split()[0]: raise ValueError(\'Manifest checksum mismatch\')\n    manifest=json.loads(manifest_bytes); verify_files(manifest[\'records\'],data_root)\n    if not torch.cuda.is_available(): raise RuntimeError(\'Enable Kaggle GPU\')\n    torch.use_deterministic_algorithms(True); torch.backends.cudnn.benchmark=False\n    device=torch.device(\'cuda\'); models={}; identities={}\n    for name,project,relative in [(\'control\',control,\'artifacts/phase3/litecrack\'),(\'boundary\',treatment,\'artifacts/phase4/litecrack_ba\')]:\n        run=project/relative\n        pointer=json.loads((run/\'checkpoints/best.json\').read_text())\n        path=(run/\'checkpoints\'/pointer[\'file\']).resolve()\n        if path.parent!=(run/\'checkpoints\').resolve(): raise ValueError(\'Unsafe checkpoint path\')\n        if hashlib.sha256(path.read_bytes()).hexdigest()!=pointer[\'sha256\']: raise ValueError(\'Checkpoint hash mismatch\')\n        state=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if state[\'manifest_hash\']!=manifest_hash: raise ValueError(\'Checkpoint split mismatch\')\n        config=state[\'config\']; model=build_model(config).to(device); model.load_state_dict(state[\'model\'])\n        expected_arch=\'litecrack\' if name==\'control\' else \'litecrack_ba\'\n        if config[\'architecture\']!=expected_arch: raise ValueError(\'Unexpected architecture\')\n        dataset=CrackDataset(manifest,data_root,\'source_validation\',config[\'image_size\'])\n        replay,_=evaluate(model,dataset,device,config[\'batch_size\'])\n        original=json.loads((run/\'evaluation.json\').read_text())[\'results\'][\'source_validation\']\n        for metric in (\'dice\',\'iou\',\'precision\',\'recall\'):\n            if abs(replay[metric]-original[metric])>1e-5: raise ValueError(\'Validation replay mismatch: \'+name)\n        models[name]=(model,config)\n        identities[name]={\'checkpoint_sha256\':pointer[\'sha256\'],\'best_epoch\':pointer[\'epoch\'],\'validation_replay\':replay}\n    output.mkdir(parents=True)\n    summary={\'manifest_sha256\':manifest_hash,\'checkpoints\':identities,\'threshold\':.5,\'splits\':{},\n             \'boundary_definition\':\'3x3 morphological gradient; replicated borders; symmetric many-to-one matching at two-pixel Chebyshev radius; both-empty=1, one-empty=0\',\n             \'scope\':\'Single-seed exploratory pilot; fixed weights, no training or threshold tuning\'}\n    for split in (\'source_validation\',\'source_holdout\',\'target_query\'):\n        rows_by_model={}; scores={}\n        for name,(model,config) in models.items():\n            dataset=CrackDataset(manifest,data_root,split,config[\'image_size\'])\n            scores[name],rows=evaluate(model,dataset,device,config[\'batch_size\'])\n            rows_by_model[name]={r[\'image_id\']:r for r in rows}\n            write_json(output/f\'{name}_{split}_per_image.json\',rows)\n        if rows_by_model[\'control\'].keys()!=rows_by_model[\'boundary\'].keys(): raise ValueError(\'Unpaired images\')\n        differences=[{\'image_id\':key,**{metric:rows_by_model[\'boundary\'][key][metric]-rows_by_model[\'control\'][key][metric]\n                     for metric in (\'dice\',\'iou\',\'precision\',\'recall\',\'boundary_f1\')}} for key in rows_by_model[\'control\']]\n        delta={metric:100*(scores[\'boundary\'][metric]-scores[\'control\'][metric]) for metric in (\'dice\',\'iou\',\'boundary_f1\')}\n        summary[\'splits\'][split]={\'scores\':scores,\'boundary_minus_control_percentage_points\':delta,\n            \'boundary_target_met_on_point_estimates\':delta[\'boundary_f1\']>=2 and delta[\'dice\']>=-1}\n        write_json(output/f\'{split}_paired_differences.json\',differences)\n        write_json(output/\'comparison.json\',summary)\n    print(json.dumps(summary,indent=2),flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'control\',\'treatment\',\'data-root\',\'output\'): p.add_argument(\'--\'+name,type=Path,required=True)\n    a=p.parse_args(); compare(a.control,a.treatment,a.data_root,a.output)\n', 'scripts/environment.py': '"""Record current interpreter and actual dependency/GPU availability."""\nimport importlib.metadata\nimport json\nimport platform\nfrom pathlib import Path\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.atomic import write_json\n\nreport = {"python": sys.version, "executable": sys.executable,\n          "platform": platform.platform(), "packages": {}}\nfor name in ("torch", "numpy", "Pillow", "pytest"):\n    try:\n        report["packages"][name] = importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        report["packages"][name] = None\ntry:\n    import torch\n    report["cuda_available"] = torch.cuda.is_available()\n    report["cuda_devices"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\nexcept ImportError:\n    report["cuda_available"] = "unverified: torch not installed"\nwrite_json("artifacts/phase1/environment.json", report)\nprint(json.dumps(report, indent=2))\n', 'scripts/evaluate_frozen_baselines.py': '"""Final baseline evaluation: fixed validation-best weights and threshold, no tuning."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport torch\nfrom src.atomic import write_json\nfrom src.dataset import CrackDataset,verify_files\nfrom src.models import SegmentationUNet\nfrom scripts.train_baseline import evaluate\n\n\ndef run_evaluation(project,data_root,output):\n    project=Path(project); output=Path(output)\n    if output.exists(): raise ValueError(\'Evaluation output exists; preserve it and use a new directory\')\n    manifest_path=project/\'data/manifests/pilot_v1.json\'\n    manifest_bytes=manifest_path.read_bytes()\n    manifest_hash=hashlib.sha256(manifest_bytes).hexdigest()\n    if manifest_hash!=manifest_path.with_suffix(\'.json.sha256\').read_text().split()[0]:\n        raise ValueError(\'Manifest checksum mismatch\')\n    manifest=json.loads(manifest_bytes)\n    verify_files(manifest[\'records\'],data_root)\n    if not torch.cuda.is_available(): raise RuntimeError(\'Use the verified Kaggle GPU runtime\')\n    output.mkdir(parents=True)\n    summary={}\n    for architecture in (\'unet\',\'r2au\'):\n        run=project/\'artifacts/phase2\'/architecture\n        pointer=json.loads((run/\'checkpoints/best.json\').read_text())\n        path=(run/\'checkpoints\'/pointer[\'file\']).resolve()\n        if path.parent!=(run/\'checkpoints\').resolve(): raise ValueError(\'Unsafe checkpoint path\')\n        if hashlib.sha256(path.read_bytes()).hexdigest()!=pointer[\'sha256\']:\n            raise ValueError(\'Checkpoint checksum mismatch\')\n        # Complete recovery files use pickle. Only load the trusted local run.\n        state=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if state[\'manifest_hash\']!=manifest_hash: raise ValueError(\'Checkpoint uses different split\')\n        config=state[\'config\']\n        device=torch.device(\'cuda\')\n        model=SegmentationUNet(architecture,config[\'base_channels\'],config.get(\'normalization\',\'none\')).to(device)\n        model.load_state_dict(state[\'model\'])\n        validation=CrackDataset(manifest,data_root,\'source_validation\',config[\'image_size\'])\n        replay,_=evaluate(model,validation,device,config[\'batch_size\'])\n        expected=json.loads((run/\'evaluation.json\').read_text())[\'results\'][\'source_validation\']\n        for key in (\'dice\',\'iou\',\'precision\',\'recall\'):\n            if abs(replay[key]-expected[key])>1e-5:\n                raise ValueError(f\'{architecture} validation replay mismatch: {key}; no query evaluation performed for this model\')\n        results={}\n        for split in (\'source_holdout\',\'target_query\'):\n            dataset=CrackDataset(manifest,data_root,split,config[\'image_size\'])\n            metrics,rows=evaluate(model,dataset,device,config[\'batch_size\'])\n            results[split]=metrics\n            write_json(output/(architecture+\'_\'+split+\'_per_image.json\'),rows)\n        summary[architecture]={\'checkpoint_sha256\':pointer[\'sha256\'],\'best_epoch\':pointer[\'epoch\'],\n            \'validation_replay\':replay,\'threshold\':.5,\'results\':results}\n        write_json(output/\'evaluation.json\',{\'manifest_sha256\':manifest_hash,\'models\':summary,\n            \'scope\':\'Fixed stabilized baselines; no training or threshold tuning; exploratory pilot query previously used in v1\'})\n        print(json.dumps({architecture:summary[architecture]},indent=2),flush=True)\n    return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--project\',type=Path,required=True)\n    p.add_argument(\'--data-root\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args(); run_evaluation(a.project,a.data_root,a.output)\n', 'scripts/evaluate_litecrack.py': '"""Evaluate the immutable Phase 3 region-only checkpoint, without any tuning."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport sys\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport numpy as np\nimport torch\nfrom PIL import Image,ImageDraw\nfrom src.atomic import write_json\nfrom src.dataset import CrackDataset,verify_files\nfrom src.model_factory import build_model\nfrom scripts.train_baseline import evaluate\n\n\n@torch.no_grad()\ndef preview(model,dataset,device,output,split):\n    indices=random.Random(42).sample(range(len(dataset)),min(6,len(dataset)))\n    sheet=Image.new(\'RGB\',(1024,280*len(indices)),\'white\')\n    draw=ImageDraw.Draw(sheet)\n    for j,index in enumerate(indices):\n        image,truth=dataset[index]\n        probability=model(image[None].to(device)).sigmoid()[0,0].cpu().numpy()\n        prediction=(probability>=.5).astype(np.uint8)*255\n        original=Image.fromarray((image.numpy().transpose(1,2,0)*255).astype(np.uint8))\n        target=Image.fromarray((truth[0].numpy()*255).astype(np.uint8))\n        mask=Image.fromarray(prediction)\n        overlay=Image.composite(Image.blend(original,Image.new(\'RGB\',original.size,\'red\'),.55),original,mask)\n        for column,view in enumerate((original,target.convert(\'RGB\'),mask.convert(\'RGB\'),overlay)):\n            sheet.paste(view.resize((256,256)),(column*256,j*280+24))\n        draw.text((2,j*280+4),dataset.rows[index][\'image_id\']+\' | image / truth / prediction / overlay\',fill=\'black\')\n        mask.save(output/f\'{split}_preview_mask_{j}.png\')\n    sheet.save(output/f\'{split}_predictions.jpg\',quality=92)\n    return [dataset.rows[index][\'image_id\'] for index in indices]\n\n\ndef run_evaluation(project,data_root,output):\n    project,output=Path(project),Path(output)\n    if output.exists(): raise ValueError(\'Output exists; use a new directory and preserve prior results\')\n    manifest_path=project/\'data/manifests/pilot_v1.json\'\n    raw_manifest=manifest_path.read_bytes()\n    manifest_hash=hashlib.sha256(raw_manifest).hexdigest()\n    if manifest_hash!=manifest_path.with_suffix(\'.json.sha256\').read_text().split()[0]:\n        raise ValueError(\'Manifest hash mismatch\')\n    manifest=json.loads(raw_manifest)\n    verify_files(manifest[\'records\'],data_root)\n    run=project/\'artifacts/phase3/litecrack\'\n    pointer=json.loads((run/\'checkpoints/best.json\').read_text())\n    path=(run/\'checkpoints\'/pointer[\'file\']).resolve()\n    if path.parent!=(run/\'checkpoints\').resolve(): raise ValueError(\'Unsafe checkpoint path\')\n    if hashlib.sha256(path.read_bytes()).hexdigest()!=pointer[\'sha256\']:\n        raise ValueError(\'Checkpoint hash mismatch\')\n    # This is the trusted snapshot from the already-verified Phase 3 archive.\n    state=torch.load(path,map_location=\'cpu\',weights_only=False)\n    if state[\'manifest_hash\']!=manifest_hash: raise ValueError(\'Checkpoint used different splits\')\n    config=state[\'config\']\n    if config[\'architecture\']!=\'litecrack\': raise ValueError(\'Expected region-only LiteCrack\')\n    if not torch.cuda.is_available(): raise RuntimeError(\'Enable the Kaggle GPU runtime\')\n    torch.use_deterministic_algorithms(True); torch.backends.cudnn.benchmark=False\n    device=torch.device(\'cuda\')\n    model=build_model(config).to(device)\n    model.load_state_dict(state[\'model\'])\n    validation=CrackDataset(manifest,data_root,\'source_validation\',config[\'image_size\'])\n    replay,_=evaluate(model,validation,device,config[\'batch_size\'])\n    recorded=json.loads((run/\'evaluation.json\').read_text())[\'results\'][\'source_validation\']\n    for metric in (\'dice\',\'iou\',\'precision\',\'recall\'):\n        if abs(replay[metric]-recorded[metric])>1e-5:\n            raise ValueError(\'Validation replay mismatch; stopping before holdout/query evaluation\')\n    output.mkdir(parents=True)\n    results,preview_ids={},{}\n    for split in (\'source_holdout\',\'target_query\'):\n        dataset=CrackDataset(manifest,data_root,split,config[\'image_size\'])\n        metrics,rows=evaluate(model,dataset,device,config[\'batch_size\'])\n        results[split]=metrics\n        write_json(output/f\'litecrack_{split}_per_image.json\',rows)\n        preview_ids[split]=preview(model,dataset,device,output,split)\n    report={\'manifest_sha256\':manifest_hash,\'architecture\':\'litecrack\',\n            \'checkpoint_sha256\':pointer[\'sha256\'],\'best_epoch\':pointer[\'epoch\'],\n            \'threshold\':.5,\'validation_replay\':replay,\'results\':results,\n            \'preview_ids\':preview_ids,\'preview_selection\':\'seed 42, six samples per split; not cherry-picked\',\n            \'scope\':\'Fixed region-only model; no training, adaptation or threshold tuning; exploratory target query previously used\'}\n    write_json(output/\'evaluation.json\',report)\n    print(json.dumps(report,indent=2),flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--project\',type=Path,required=True)\n    p.add_argument(\'--data-root\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args(); run_evaluation(a.project,a.data_root,a.output)\n', 'scripts/freeze_pilot.py': '"""Freeze reviewed audit evidence into a portable, checksummed pilot protocol."""\nimport argparse\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport re\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.atomic import write_json\n\n\ndef freeze(audit_directory, output):\n    audit_directory, output = Path(audit_directory), Path(output)\n    if output.exists():\n        raise ValueError(\'Frozen protocol exists; choose a new version\')\n    source = audit_directory/\'candidate_manifest.json\'\n    candidate = json.loads(source.read_text())\n    if candidate[\'errors\']:\n        raise ValueError(\'Audit has unresolved errors\')\n    records=[]\n    ids, hashes = set(), set()\n    groups=defaultdict(set)\n    captures=defaultdict(set)\n    for original in candidate[\'records\']:\n        row=dict(original)\n        key=original[\'image_id\'].split(\':\',1)[1]\n        identity=f"{row[\'dataset\']}:{row[\'partition\']}:{key}"\n        if identity in ids or row[\'pixel_sha256\'] in hashes:\n            raise ValueError(\'Duplicate canonical ID or decoded image\')\n        ids.add(identity); hashes.add(row[\'pixel_sha256\'])\n        groups[row[\'group_id\']].add(row[\'split\'])\n        if row[\'dataset\']==\'CrackMap\':\n            match=re.fullmatch(r\'(GOPR\\d+)(?: \\(\\d+\\))?\',key)\n            if not match: raise ValueError(\'Unexpected CrackMap capture naming\')\n            captures[match.group(1)].add(row[\'split\'])\n        derived=audit_directory/\'derived_masks\'/row[\'dataset\']/row[\'partition\']/(key+\'.png\')\n        if hashlib.sha256(derived.read_bytes()).hexdigest()!=row[\'mask_sha256\']:\n            raise ValueError(\'Derived mask checksum mismatch\')\n        def relative_raw(path):\n            marker=\'/data/raw/\'\n            if marker not in path: raise ValueError(\'Unknown raw path layout\')\n            return path.split(marker,1)[1]\n        records.append({\n            \'image_id\':identity,\'original_id\':original[\'image_id\'],\n            \'dataset\':row[\'dataset\'],\'partition\':row[\'partition\'],\'split\':row[\'split\'],\n            \'group_id\':row[\'group_id\'],\'group_provenance\':row[\'group_provenance\'],\n            \'image\':relative_raw(row[\'image\']), \'mask\':relative_raw(row[\'raw_mask\']),\n            \'image_sha256\':row[\'image_sha256\'],\'mask_sha256\':row[\'raw_mask_sha256\'],\n            \'pixel_sha256\':row[\'pixel_sha256\'],\'width\':row[\'width\'],\'height\':row[\'height\'],\n            \'foreground_fraction\':row[\'foreground_fraction\'],\'mask_values\':row[\'mask_values\']})\n    if any(len(v)!=1 for v in groups.values()) or any(len(v)!=1 for v in captures.values()):\n        raise ValueError(\'Group/capture leakage\')\n    counts=dict(sorted(Counter(r[\'split\'] for r in records).items()))\n    expected={\'source_train\',\'source_validation\',\'source_holdout\',\'target_acquisition\',\'target_query\'}\n    if set(counts)!=expected:\n        raise ValueError(\'Missing required splits\')\n    # A shot is one image, with a distinct GOPR capture per shot in each draw.\n    support=defaultdict(list)\n    for row in records:\n        if row[\'split\']==\'target_acquisition\': support[row[\'group_id\']].append(row[\'image_id\'])\n    if len(support)<10: raise ValueError(\'Fewer than ten support capture groups\')\n    draws=[]\n    for draw in range(5):\n        rng=random.Random(42+draw)\n        selected=rng.sample(sorted(support),10)\n        images=[rng.choice(sorted(support[group])) for group in selected]\n        draws.append({\'draw\':draw,\'seed\':42+draw,\'supports\':{str(k):images[:k] for k in (1,5,10)}})\n    result={\'schema_version\':1,\'protocol\':\'deepcrack_crackmap_pilot_v1\',\n        \'status\':\'FROZEN_PILOT\',\'seed\':42,\'path_base\':\'data/raw\',\n        \'counts\':counts,\'records\':records,\'support_draws\':draws,\n        \'audit_manifest_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\n        \'mask_policy\':\'White foreground; binary 0/255; RGB labels require identical channels.\',\n        \'review\':\'24 contact-sheet examples inspected locally; not every pair visually reviewed.\',\n        \'limitations\':[\n            \'Pilot uses DeepCrack only as source, not the reference paper five-source protocol.\',\n            \'CrackMap GOPR filename captures stay in one split; road/site/sequence independence is unverified.\',\n            \'DeepCrack official train/test retained; no verified capture groups beyond similarity screening.\',\n            \'dHash screening found no matches but cannot rule out related captures.\',\n            \'CrackMap license terms remain unconfirmed; protocol freezing is not license clearance.\',\n            \'Ten reused DeepCrack filename IDs qualified by native partition; images have distinct hashes.\',\n            \'Raw images remain on Kaggle; local artifact checks verify exported evidence, not a full image re-audit.\',\n            \'No model training or GPU trainer recovery has run.\']}\n    write_json(output,result)\n    digest=hashlib.sha256(output.read_bytes()).hexdigest()\n    output.with_suffix(\'.json.sha256\').write_text(digest+\'  \'+output.name+\'\\n\')\n    print(json.dumps({\'manifest\':str(output),\'sha256\':digest,\'counts\':counts},indent=2))\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'audit_directory\',type=Path)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args()\n    freeze(args.audit_directory,args.output)\n', 'scripts/prepare_data.py': '"""Audit paired images and create an immutable, group-separated pilot manifest.\n\nInput CSV columns: image_id,image,mask,dataset,group_id,partition.\nPaths are relative to the CSV. partition is source_train, source_test, or target.\nUse explicit parent-frame/sequence group IDs; filenames alone do not establish\nindependence. Missing metadata is a blocker to freezing, not silently invented.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.atomic import write_json\n\n\ndef build_manifest(csv_path, seed=42):\n    import numpy as np\n    from PIL import Image\n    csv_path = Path(csv_path).resolve()\n    records, seen_ids, seen_pixels, group_partitions = [], set(), set(), {}\n    with csv_path.open(newline="", encoding="utf-8-sig") as stream:\n        for row in csv.DictReader(stream):\n            identity = (row["dataset"], row["image_id"])\n            if identity in seen_ids or not row["group_id"].strip():\n                raise ValueError(f"Duplicate ID or missing group: {identity}")\n            seen_ids.add(identity)\n            partition = row["partition"]\n            if partition not in {"source_train", "source_test", "target"}:\n                raise ValueError(f"Invalid partition: {partition}")\n            group = row["dataset"] + ":" + row["group_id"]\n            if group in group_partitions and group_partitions[group] != partition:\n                raise ValueError(f"Group spans partitions: {group}")\n            group_partitions[group] = partition\n            image_path = (csv_path.parent / row["image"]).resolve()\n            mask_path = (csv_path.parent / row["mask"]).resolve()\n            with Image.open(image_path) as original:\n                image = original.convert("RGB")\n                image.load()\n            with Image.open(mask_path) as original:\n                # Do not silently turn colored labels into binary targets.\n                if original.mode not in {"1", "L"}:\n                    raise ValueError(f"Mask needs explicit grayscale conversion: {mask_path}")\n                mask = np.asarray(original.convert("L"))\n            if image.size != (mask.shape[1], mask.shape[0]):\n                raise ValueError(f"Image/mask size mismatch: {identity}")\n            values = set(np.unique(mask).tolist())\n            if not values <= {0, 255}:\n                raise ValueError(f"Mask must use 0/255, found {values}: {identity}")\n            pixel_hash = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()\n            if pixel_hash in seen_pixels:\n                raise ValueError(f"Duplicate decoded image: {identity}")\n            seen_pixels.add(pixel_hash)\n            records.append({**row, "image": str(image_path), "mask": str(mask_path),\n                            "group": group, "width": image.width, "height": image.height,\n                            "image_sha256": hashlib.sha256(image_path.read_bytes()).hexdigest(),\n                            "mask_sha256": hashlib.sha256(mask_path.read_bytes()).hexdigest(),\n                            "pixel_sha256": pixel_hash, "foreground_fraction": float((mask > 0).mean())})\n    if not records:\n        raise ValueError("No image/mask pairs")\n    rng = random.Random(seed)\n    allocations = {}\n    for partition, first, second, fraction in [\n        ("source_train", "source_validation", "source_train", .2),\n        ("target", "target_acquisition", "target_query", .25),\n    ]:\n        groups = sorted({r["group"] for r in records if r["partition"] == partition})\n        if len(groups) < 2:\n            raise ValueError(f"Need at least two groups in {partition}")\n        rng.shuffle(groups)\n        count = max(1, min(len(groups)-1, round(len(groups)*fraction)))\n        allocations.update({g: first if i < count else second for i, g in enumerate(groups)})\n    for row in records:\n        row["split"] = "source_holdout" if row["partition"] == "source_test" else allocations[row["group"]]\n    counts = {s: sum(r["split"] == s for r in records) for s in sorted({r["split"] for r in records})}\n    return {"schema_version": 1, "seed": seed, "counts": counts,\n            "limitations": ["Exact decoded duplicates checked; near-duplicate/site independence requires manual audit.",\n                            "Split fractions apply to groups; image fractions may differ."], "records": records}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("index", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.output.exists():\n        raise SystemExit("Refusing to overwrite a frozen manifest; choose a new version")\n    result = build_manifest(args.index, args.seed)\n    write_json(args.output, result)\n    print(json.dumps(result["counts"], indent=2))\n    print("Manifest SHA256:", hashlib.sha256(args.output.read_bytes()).hexdigest())\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/profile_baseline.py': '"""Profile validation-best weights; does not use query labels or change training."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport statistics\nimport sys\nimport time\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport torch\nfrom src.atomic import write_json\nfrom src.model_factory import build_model\n\n\ndef conv_linear_macs(model,image):\n    total=[0]\n    handles=[]\n    def count(module,inputs,output):\n        if isinstance(module,torch.nn.Conv2d):\n            total[0]+=output.numel()*(module.in_channels//module.groups)*module.kernel_size[0]*module.kernel_size[1]\n        elif isinstance(module,torch.nn.Linear):\n            total[0]+=output.numel()*module.in_features\n    for module in model.modules():\n        if isinstance(module,(torch.nn.Conv2d,torch.nn.Linear)):\n            handles.append(module.register_forward_hook(count))\n    try:\n        with torch.inference_mode(): model(image)\n    finally:\n        for handle in handles: handle.remove()\n    return total[0]\n\n\ndef profile(run_dir,device_name):\n    run=Path(run_dir)\n    pointer=json.loads((run/\'checkpoints/best.json\').read_text())\n    checkpoint=run/\'checkpoints\'/pointer[\'file\']\n    assert checkpoint.resolve().parent==(run/\'checkpoints\').resolve()\n    assert hashlib.sha256(checkpoint.read_bytes()).hexdigest()==pointer[\'sha256\']\n    state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    config=state[\'config\']\n    device=torch.device(device_name)\n    torch.set_num_threads(4)\n    model=build_model(config).to(device).eval()\n    model.load_state_dict(state[\'model\'])\n    image=torch.zeros(1,3,config[\'image_size\'],config[\'image_size\'],device=device)\n    def sync():\n        if device.type==\'cuda\': torch.cuda.synchronize(device)\n    with torch.inference_mode():\n        for _ in range(20): model(image)\n        sync()\n        if device.type==\'cuda\': torch.cuda.reset_peak_memory_stats(device)\n        samples=[]\n        for _ in range(200):\n            sync(); start=time.perf_counter(); model(image); sync()\n            samples.append((time.perf_counter()-start)*1000)\n    report={\'architecture\':config[\'architecture\'],\'device\':str(device),\n            \'hardware\':torch.cuda.get_device_name(device) if device.type==\'cuda\' else __import__(\'platform\').processor(),\n            \'threads\':torch.get_num_threads(),\'batch_size\':1,\'image_size\':config[\'image_size\'],\n            \'parameters\':sum(p.numel() for p in model.parameters()),\n            \'p50_ms\':statistics.median(samples),\'p95_ms\':sorted(samples)[189],\n            \'warmups\':20,\'timed_iterations\':200,\'scope\':\'model pass only; no image decoding/transfer\',\n            \'recovery_checkpoint_bytes\':checkpoint.stat().st_size,\n            \'fp32_weight_bytes\':sum(p.numel()*p.element_size() for p in model.state_dict().values()),\n            \'cuda_peak_allocated_bytes\':torch.cuda.max_memory_allocated(device) if device.type==\'cuda\' else None,\n            \'conv_linear_macs\':conv_linear_macs(model,image),\n            \'mac_convention\':\'One MAC per multiply-accumulate; Conv2d/Linear only; excludes normalization, activation, pooling, resize and elementwise operations\'}\n    write_json(run/f\'profile_{device.type}.json\',report)\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--run-dir\',type=Path,required=True)\n    p.add_argument(\'--device\',choices=[\'cpu\',\'cuda\'],required=True)\n    a=p.parse_args(); profile(a.run_dir,a.device)\n', 'scripts/restore_datasets.py': '"""Recover pinned datasets without changing frozen splits or overwriting data."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom scripts.audit_kaggle_data import source_layout\nfrom src.atomic import write_json\n\n\ndef verify_inventory(root,records):\n    root=Path(root).resolve()\n    for row in records:\n        for field in (\'image\',\'mask\'):\n            path=(root/row[field]).resolve()\n            if not path.is_relative_to(root): raise ValueError(\'Manifest path escapes data root\')\n            if not path.is_file(): raise FileNotFoundError(str(path))\n            if hashlib.sha256(path.read_bytes()).hexdigest()!=row[field+\'_sha256\']:\n                raise ValueError(f"Dataset checksum mismatch: {row[\'image_id\']} {field}; original preserved")\n\n\ndef complete(root,records):\n    return all((root/r[field]).is_file() for r in records for field in (\'image\',\'mask\'))\n\n\ndef restore(manifest_path,destination,sources_path,search_roots=()):\n    manifest_path=Path(manifest_path)\n    expected=manifest_path.with_suffix(\'.json.sha256\').read_text().split()[0]\n    if hashlib.sha256(manifest_path.read_bytes()).hexdigest()!=expected:\n        raise ValueError(\'Frozen manifest checksum mismatch\')\n    manifest=json.loads(manifest_path.read_text())\n    records=manifest[\'records\']\n    destination=Path(destination).resolve()\n    candidates=[destination]+[Path(p).resolve() for p in search_roots]\n    input_root=Path(\'/kaggle/input\')\n    if input_root.exists():\n        candidates.extend(p.parent for p in input_root.rglob(\'DeepCrack\') if p.is_dir())\n    for candidate in dict.fromkeys(candidates):\n        if complete(candidate,records):\n            verify_inventory(candidate,records)\n            print(\'Reused existing data; all image/mask checksums passed:\',candidate)\n            return candidate\n    print(\'Previous data unavailable/incomplete. Restoring pinned repositories to:\',destination,flush=True)\n    destination.mkdir(parents=True,exist_ok=True)\n    sources=json.loads(Path(sources_path).read_text())\n    for name,info in sources.items():\n        folder=destination/name\n        if not folder.exists():\n            result=subprocess.run([\'git\',\'clone\',\'--no-checkout\',info[\'url\'],str(folder)],\n                                  capture_output=True,text=True)\n            if result.returncode:\n                raise RuntimeError(f\'{name} download failed. Enable Kaggle Internet or attach the original datasets. \'+result.stderr[-2000:])\n            subprocess.run([\'git\',\'-C\',str(folder),\'checkout\',\'--detach\',info[\'commit\']],check=True)\n        elif not (folder/\'.git\').exists():\n            print(\'Existing non-Git folder preserved:\',folder)\n        else:\n            actual=subprocess.check_output([\'git\',\'-C\',str(folder),\'rev-parse\',\'HEAD\'],text=True).strip()\n            if actual!=info[\'commit\']:\n                raise ValueError(f\'{name} is at a different commit; preserve it and choose a new destination\')\n    source_layout(destination)  # Safely unpack DeepCrack; original archive retained.\n    verify_inventory(destination,records)\n    print(\'Restoration complete: verified\',len(records),\'pairs against the unchanged frozen manifest\')\n    return destination\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--manifest\',type=Path,required=True)\n    p.add_argument(\'--destination\',type=Path,required=True)\n    p.add_argument(\'--sources\',type=Path,required=True)\n    p.add_argument(\'--search-root\',type=Path,action=\'append\',default=[])\n    p.add_argument(\'--report\',type=Path,required=True)\n    a=p.parse_args()\n    root=restore(a.manifest,a.destination,a.sources,a.search_root)\n    write_json(a.report,{\'data_root\':str(root),\'verified\':True,\'pairs\':len(json.loads(a.manifest.read_text())[\'records\'])})\n', 'scripts/train_baseline.py': '"""Deterministic single-device baseline training, resumable after optimizer steps."""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport signal\nimport shutil\nimport sys\nimport time\n\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\',\':4096:8\')\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport numpy as np\nimport torch\nfrom src.atomic import write_json\nfrom src.checkpoints import save_checkpoint,load_checkpoint\nfrom src.dataset import CrackDataset,verify_files\nfrom src.model_factory import build_model\nfrom src.segmentation import dice_loss,image_metrics,region_loss\nfrom src.boundaries import boundary_loss,boundary_f1\n\n\ndef source_hash():\n    root=Path(__file__).resolve().parents[1]\n    names=[\'src/models.py\',\'src/litecrack.py\',\'src/boundaries.py\',\'src/model_factory.py\',\'src/dataset.py\',\'src/segmentation.py\',\'src/checkpoints.py\',\'scripts/train_baseline.py\']\n    return hashlib.sha256(b\'\'.join((root/n).read_bytes() for n in names)).hexdigest()\n\n\n@torch.no_grad()\ndef evaluate(model,dataset,device,batch_size):\n    model.eval(); totals={key:0. for key in (\'dice\',\'iou\',\'precision\',\'recall\',\'boundary_f1\')}\n    per_image=[]; loss_sum=0; predicted_fraction=0.; truth_fraction=0.\n    for start in range(0,len(dataset),batch_size):\n        indices=list(range(start,min(start+batch_size,len(dataset))))\n        samples=[dataset[i] for i in indices]\n        x=torch.stack([s[0] for s in samples]).to(device)\n        y=torch.stack([s[1] for s in samples]).to(device)\n        logits=model(x)\n        if isinstance(logits,tuple): logits=logits[0]\n        predicted_fraction+=float((logits>=0).float().mean())*len(indices)\n        truth_fraction+=float(y.mean())*len(indices)\n        loss_sum+=float(dice_loss(logits,y))*len(indices)\n        values=image_metrics(logits,y)\n        values[\'boundary_f1\']=boundary_f1(logits,y)\n        for j,index in enumerate(indices):\n            record={\'image_id\':dataset.rows[index][\'image_id\']}\n            for key,vector in values.items():\n                value=float(vector[j]); totals[key]+=value; record[key]=value\n            per_image.append(record)\n    return {\'loss\':loss_sum/len(dataset),**{k:v/len(dataset) for k,v in totals.items()},\n            \'predicted_foreground_fraction\':predicted_fraction/len(dataset),\n            \'truth_foreground_fraction\':truth_fraction/len(dataset),\'n\':len(dataset)},per_image\n\n\ndef train(config,manifest_path,data_root,run_dir,resume=False,stop_after_steps=None,final_evaluation=True):\n    run=Path(run_dir)\n    if not resume and run.exists() and any(run.iterdir()):\n        raise ValueError(\'Run directory is not empty; use --resume or a new run\')\n    run.mkdir(parents=True,exist_ok=True)\n    manifest_path=Path(manifest_path)\n    manifest=json.loads(manifest_path.read_text())\n    if manifest.get(\'status\')!=\'FROZEN_PILOT\': raise ValueError(\'Expected frozen pilot protocol\')\n    manifest_hash=hashlib.sha256(manifest_path.read_bytes()).hexdigest()\n    config={**config,\'source_sha256\':source_hash(),\'torch_version\':torch.__version__,\n            \'cuda_runtime\':torch.version.cuda,\n            \'torchvision_version\':importlib.metadata.version(\'torchvision\') if config[\'architecture\'].startswith(\'litecrack\') else None}\n    if config.get(\'loss\') not in (\'dice\',\'bce_dice\'): raise ValueError(\'Unknown loss\')\n    if config[\'batch_size\']<1 or config[\'checkpoint_every_steps\']<1 or config[\'image_size\']<16:\n        raise ValueError(\'Invalid batch/checkpoint frequency/image size\')\n    device=torch.device(config[\'device\'])\n    if device.type==\'cuda\' and not torch.cuda.is_available(): raise RuntimeError(\'CUDA unavailable\')\n    if config[\'deterministic\']:\n        torch.use_deterministic_algorithms(True)\n        torch.backends.cudnn.benchmark=False\n    random.seed(config[\'seed\']); np.random.seed(config[\'seed\']); torch.manual_seed(config[\'seed\'])\n    if device.type==\'cuda\': torch.cuda.manual_seed_all(config[\'seed\'])\n    # Verify all hashes once before training; target data is never used for updates.\n    verify_files(manifest[\'records\'],data_root)\n    training=CrackDataset(manifest,data_root,\'source_train\',config[\'image_size\'],True)\n    validation=CrackDataset(manifest,data_root,\'source_validation\',config[\'image_size\'])\n    model=build_model(config).to(device)\n    if config.get(\'max_parameters\') and sum(p.numel() for p in model.parameters())>config[\'max_parameters\']:\n        raise ValueError(\'Compact-model parameter budget exceeded\')\n    optimizer=torch.optim.Adam(model.parameters(),lr=config[\'learning_rate\'])\n    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode=\'min\',patience=5,factor=.1)\n    epoch=step=cursor=bad_epochs=0; order=None; best=-1.; history=[]\n    loss_total=0.; seen=0\n    if resume:\n        state=load_checkpoint(run/\'checkpoints\',model=model,optimizer=optimizer,scheduler=scheduler,\n            scaler=None,expected_manifest_hash=manifest_hash,expected_config=config)\n        epoch,step,best=state[\'epoch\'],state[\'global_step\'],state[\'best_metric\']\n        sampler=state[\'sampler_state\']\n        order,cursor,bad_epochs=sampler[\'order\'],sampler[\'cursor\'],sampler[\'bad_epochs\']\n        history=sampler[\'history\']; loss_total=sampler[\'loss_total\']; seen=sampler[\'seen\']\n    else:\n        write_json(run/\'config.json\',config)\n        write_json(run/\'manifest.json\',manifest)\n    write_json(run/\'environment.json\',{\'python\':sys.version,\'torch\':torch.__version__,\n        \'device\':str(device),\'gpu\':torch.cuda.get_device_name(device) if device.type==\'cuda\' else None,\n        \'parameters\':sum(p.numel() for p in model.parameters())})\n    stopping={\'requested\':False}\n    def request_stop(*_): stopping[\'requested\']=True\n    old_handlers={}\n    for sig in (signal.SIGINT,signal.SIGTERM):\n        old_handlers[sig]=signal.signal(sig,request_stop)\n    def snapshot(is_best=False):\n        # Do not delete earlier snapshots to make space. Leave recoverable files\n        # intact and fail explicitly before a likely out-of-space write.\n        estimated=sum(p.numel()*p.element_size() for p in model.parameters())*4\n        if shutil.disk_usage(run).free < estimated+256*1024**2:\n            raise RuntimeError(\'Insufficient checkpoint disk space; export results and resume on persistent storage\')\n        save_checkpoint(run/\'checkpoints\',model=model,optimizer=optimizer,scheduler=scheduler,\n            scaler=None,epoch=epoch,global_step=step,\n            sampler_state={\'order\':order,\'cursor\':cursor,\'bad_epochs\':bad_epochs,\n                           \'history\':history,\'loss_total\':loss_total,\'seen\':seen},\n            config=config,manifest_hash=manifest_hash,metrics=history[-1] if history else {},\n            best_metric=best,is_best=is_best)\n    started=time.monotonic()\n    try:\n        while epoch<config[\'epochs\'] and bad_epochs<config[\'patience\']:\n            if order is None:\n                order=torch.randperm(len(training)).tolist(); cursor=0; loss_total=0.; seen=0\n            model.train()\n            while cursor<len(order):\n                indices=order[cursor:cursor+config[\'batch_size\']]\n                samples=[training[i] for i in indices]\n                x=torch.stack([s[0] for s in samples]).to(device)\n                y=torch.stack([s[1] for s in samples]).to(device)\n                optimizer.zero_grad(set_to_none=True)\n                prediction=model(x)\n                if isinstance(prediction,tuple):\n                    loss=region_loss(prediction[0],y,config[\'loss\'])+config.get(\'boundary_weight\',.2)*boundary_loss(prediction[1],y)\n                else:\n                    loss=region_loss(prediction,y,config[\'loss\'])\n                if not torch.isfinite(loss): raise RuntimeError(\'Nonfinite training loss; inspect run\')\n                loss.backward()\n                if config.get(\'gradient_clip_norm\'):\n                    torch.nn.utils.clip_grad_norm_(model.parameters(),config[\'gradient_clip_norm\'],error_if_nonfinite=True)\n                optimizer.step()\n                cursor+=len(indices); step+=1\n                loss_total+=float(loss.detach())*len(indices); seen+=len(indices)\n                stop=stopping[\'requested\'] or (stop_after_steps is not None and step>=stop_after_steps)\n                if step%config[\'checkpoint_every_steps\']==0 or stop: snapshot()\n                if stop:\n                    write_json(run/\'status.json\',{\'state\':\'stopped_safely\',\'epoch\':epoch,\'step\':step,\n                                                  \'elapsed_seconds_this_session\':time.monotonic()-started})\n                    return\n            metrics,_=evaluate(model,validation,device,config[\'batch_size\'])\n            scheduler.step(metrics[\'loss\'])\n            improved=metrics[\'dice\']>best\n            if improved: best=metrics[\'dice\']; bad_epochs=0\n            else: bad_epochs+=1\n            history.append({\'epoch\':epoch+1,\'step\':step,\'train_loss\':loss_total/seen,\n                            \'validation\':metrics,\'learning_rate\':optimizer.param_groups[0][\'lr\']})\n            print(json.dumps(history[-1]),flush=True)\n            epoch+=1; order=None; cursor=0; loss_total=0.; seen=0\n            snapshot(improved)\n            # Recovery history is authoritative; replace logs rather than append duplicates.\n            write_json(run/\'history.json\',history)\n        write_json(run/\'status.json\',{\'state\':\'training_complete\',\'epoch\':epoch,\'step\':step,\n            \'best_validation_dice\':best,\'elapsed_seconds_this_session\':time.monotonic()-started})\n        if final_evaluation:\n            load_checkpoint(run/\'checkpoints\',model=model,optimizer=optimizer,scheduler=scheduler,\n                scaler=None,expected_manifest_hash=manifest_hash,expected_config=config,which=\'best\')\n            final={}\n            for split in config.get(\'evaluation_splits\',[\'source_holdout\',\'target_query\']):\n                dataset=CrackDataset(manifest,data_root,split,config[\'image_size\'])\n                metrics,images=evaluate(model,dataset,device,config[\'batch_size\'])\n                final[split]=metrics\n                write_json(run/(split+\'_per_image.json\'),images)\n            write_json(run/\'evaluation.json\',{\'checkpoint\':\'validation_best\',\'threshold\':.5,\'results\':final})\n    except Exception as error:\n        # Do not save partially executed optimizer steps over valid recovery state.\n        write_json(run/\'failure.json\',{\'error\':str(error),\'epoch\':epoch,\'last_step\':step})\n        raise\n    finally:\n        for sig,handler in old_handlers.items(): signal.signal(sig,handler)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--config\',required=True,type=Path)\n    p.add_argument(\'--manifest\',required=True,type=Path)\n    p.add_argument(\'--data-root\',required=True,type=Path)\n    p.add_argument(\'--run-dir\',required=True,type=Path)\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--stop-after-steps\',type=int)\n    a=p.parse_args()\n    train(json.loads(a.config.read_text()),a.manifest,a.data_root,a.run_dir,a.resume,a.stop_after_steps)\n', 'configs/data_sources.json': '{\n  "DeepCrack": {\n    "url": "https://github.com/yhlleo/DeepCrack.git",\n    "commit": "8202a60701068645c883ed68b95fd3f30914d90c"\n  },\n  "CrackMap": {\n    "url": "https://github.com/ikatsamenis/CrackMap.git",\n    "commit": "bd8fbd7789993cf278e6118836071f2ed32acffd"\n  }\n}\n'}
specs=[{'phase': 3, 'archive': 'phase3_results.zip', 'sha256': 'd1d9c4fa7c42efb64e2f30f775fbbf8f5973f22676f591ba1537523d63119470', 'hashes': {'artifacts/dataset_restore.json': '214198c56e4b3668f1de4e535a35130828097052bf3914e6eef5c14260aa1d73', 'artifacts/phase1/environment.json': '0c297add6404b6a1035b71c2cec0737c3af9b6b2ebd8a82f8e8cc81844ed688b', 'artifacts/phase3/litecrack/checkpoints/best.json': '2dfdcca8668156f7647238ae34e971b7552b65a6be74b9aceb66f163b225512b', 'artifacts/phase3/litecrack/checkpoints/latest.json': '478e85657f08c3b032713e8492081797a21917688c454c7ced3b665e8e9f3767', 'artifacts/phase3/litecrack/checkpoints/step-000002970-92069633.pt': '9ac52e68c86938f826ddf9bf5176ce8abba830c4ecb3cdd01dd35823777c1fa2', 'artifacts/phase3/litecrack/checkpoints/step-000003000-79b0a41a.pt': '1d88d2f70b9f44470fe45dcf2834ba21f6a5e802ad8dc1df6e9d44425512e198', 'artifacts/phase3/litecrack/config.json': 'c607602bba78dde827a477e5b1613b38d0a7808645e8ae66311e7f7f977b7e56', 'artifacts/phase3/litecrack/environment.json': '50f5834685be708a6c6d7b679c6a56b7a546e6eb079a0db8edcc842c4129bf9c', 'artifacts/phase3/litecrack/evaluation.json': 'd21ee6e0825963caa13b47810bcd2e336e77f88681f3776cc4c5177f45b47058', 'artifacts/phase3/litecrack/history.json': '5abccf307b6d5e13fd55d7aa62503859e0927e4a7b773ca884e8eff4161d52ef', 'artifacts/phase3/litecrack/manifest.json': '9c97554166af5494c746d46885425f748ea46c4035863c22c90215b3506358ae', 'artifacts/phase3/litecrack/profile_cpu.json': 'a89a9dbc72a39d8d05fced673cf0c89d254e69c6cf13e47e17041b46b00ac337', 'artifacts/phase3/litecrack/profile_cuda.json': '8db72d925f37716db44d672264397eee6a7501213f2a947723fdd5dea4a732dc', 'artifacts/phase3/litecrack/source_validation_per_image.json': 'd1b514ed32da6892e919be519d5bc61b402d1121fbcaa199aa468b5eb2d4c5fd', 'artifacts/phase3/litecrack/status.json': 'ba249456a441ed9c618bc77ca6a30eb70a9309062a95a70e2e49345bbdff84e3', 'artifacts/phase3/profile_cpu.log': '119ee0f32c7719a7bcd507432e826bb9fa3a76ea010764e8a8effc34fc29a663', 'artifacts/phase3/profile_cuda.log': 'bf21906b30a576839d2e6cd1c48fd36b33f93038756ba3aa4db53c7b9b54e8ec', 'artifacts/phase3/requirements.kaggle.lock.txt': '75abf5a5b84e04e2e3ba62cbad028024968b3336d10544c8479fe9e5f0918313', 'artifacts/phase3/tests.txt': '35af24e44967e6edfec2374d3ce17a9c38dee713b50aacdeb2608a1d89ee03ca', 'artifacts/phase3/tests.xml': '476d93d2bfea2af96fdf327ab2c8e713363499f7a79419c05397980d7ae0233d', 'artifacts/phase3/training.log': '96e39705f42bd47b60aee4b3b47dbebdcedb61fbffdfe8a524cb751051c0cdb7', 'configs/baseline_r2au.json': '76f058a412db7f81d4b925223f3e91b53ae5d1b893dda58f65974bb4151d0577', 'configs/baseline_unet.json': '205b44ada0cf895862f3bb31a33041b786b1eed24c880c38b9c083f475c9659c', 'configs/data_sources.json': 'e33ad521df49827fc30af7308541af4fb2a2b69ccd34ac919c19d2ddc5861c1f', 'configs/litecrack.json': 'fc99fafbcd4e4e5c5cf50d57d027271e71b0b0f4830af7196ab896a20c925fa4', 'configs/phase1.json': '51484e20e224486e7db55fab2ed8a0ded307cf2962fd53467c4b5311c1caf1e4', 'configs/stabilized_r2au.json': '197ffd9b58eec40d9e3e5de225e6890d869424e5900bbe2471dd15535a020be3', 'configs/stabilized_unet.json': 'fcd8006ecbd6c4ecbfcbb33c850c7736868a443dd95aa5551a3edc8bf2d03e08', 'data/manifests/baseline_registry_v1.json': '365e9d8c74d39e6bebf6f82c980a0148f5708bbfc17886adc688edca9e8c1f21', 'data/manifests/pilot_v1.json': '9c97554166af5494c746d46885425f748ea46c4035863c22c90215b3506358ae', 'data/manifests/pilot_v1.json.sha256': 'e04269dae5566234d83b5c20a6c631baaba57092fbb4e62dcf0f910c3bb05364', 'requirements.txt': '6f109fbff5a330a6e0f57b1a687e466ab0b330fa3921774e5e0310e65e3ee512', 'scripts/audit_kaggle_data.py': '463dce488a03d7a432df6c16bba74ae619fbf41b08b413e27aa90e6e2fa222da', 'scripts/build_data_audit_notebook.py': '68c90fa0accc72b22022ed11994d038f1812a7cb19f295d84ccadf50d1029ba0', 'scripts/build_final_evaluation_notebook.py': 'caf2d85b5d6d1b57eacd4f43bcd7116382e84a169c99acfa3013d8975b4098f2', 'scripts/build_kaggle_notebook.py': 'd121b7956103c42e9f2b340387286f623172e8e01792ddcaa358541a468af79d', 'scripts/build_phase2_notebook.py': 'dc05c84ebad18b9f3828f78563bb75bf2d954eea3874d428b68eb36e9390d8e0', 'scripts/environment.py': 'e0386ae0d9a9ef13a5fb27d3464b2afa2faa1181a055fbaf3b9aae48702183c3', 'scripts/evaluate_frozen_baselines.py': '864af8feb41d08786629a4f0f2ef4aea8f7c69e365e60ef8fcc3830890f1f6b8', 'scripts/freeze_pilot.py': 'bf7edba7258f0ab9750ff1197b12cce850f71b0d278de3fa353399b9ad45de85', 'scripts/prepare_data.py': 'fc59ddd85cde50d1e30d7d988ad8c2f898c11deb0223119e66149a67c3e54c25', 'scripts/profile_baseline.py': 'f62e6f9e121255193add4ef575d5c77d2503b02337dcacac4e483cdc445faa41', 'scripts/restore_datasets.py': '0d7fc8fd923c817c1da393abe2b76a4b49625a433f8724a88b62bd97f2791a5b', 'scripts/train_baseline.py': '2e1ec42b0d7f62ae92553d1f65096f252918a3d826a5d33ce2f96e443b6c09ec', 'src/__init__.py': '8ccf2c63120e3ea084ae3827cd5d4189d092f266d9ce6e11a97fb2b5aa175bef', 'src/atomic.py': '8d3e04bad91ed969540d145613c83b719c69f3346bf5ced3bda46dec5be84823', 'src/checkpoints.py': '0a34e05d17f4b4d255abdc48b38b6c429a6427ac8fe351d9a1204d40bd783264', 'src/dataset.py': 'a472d7c679edd5fadb974b8ef168db29ddcf2288a2beca23345a0e3379fafc57', 'src/litecrack.py': '2c944b12a9f95d19284a2478a1457d53c7af6fac02601761c19bea3f9384b6d9', 'src/model_factory.py': '890fdc511753cd6357caeeccbe2a125878d223fb25607223a34b0e7f76198110', 'src/models.py': '011980f40fc4c97b2a42a39048231d10e157e5e38bb70ffd6ef33f27de1214e5', 'src/segmentation.py': '9016f5ba4cb859c9f448e3889b0d8f04cd3a3f5848e5ca59898db96cbdb082b5', 'tests/test_checkpoint_resume.py': 'fc852c67163aaee31d4cb93b64b4c077fc888d52ad5eb99778553c3287f0a56e', 'tests/test_data_audit.py': '222b4de7fb5c2efed6183b95c90d8b0ca173c3e59c1ef19d9a4362cd3ab0d277', 'tests/test_dataset_restore.py': 'f480b83820f37077d605546b692483af8dbf844b3347a494bd7c28fe6f4ac7c4', 'tests/test_foundation.py': 'b511f320a296782b354c4169d4fff15428180dbc436aede61aa52b2c2c778c39', 'tests/test_litecrack.py': '9d2d3021527adbbf8a60a1a5ba9bc43f5b0b60fb3933342db41ec397b8fcb3e3', 'tests/test_pair_discovery.py': '78873635f09924fcd8f0174dda4cf8621576f22499807ab60b665a3f74cff440', 'tests/test_phase2.py': 'c4e23a1ce0a5df9fdcef14342ed574e456f5abc0b72623799ed6935625a2b903'}}, {'phase': 4, 'archive': 'phase4_results.zip', 'sha256': '0851dc03b170e41df2639432a2373579e594a7398f8e0c78183eb5f2a86d652c', 'hashes': {'artifacts/dataset_restore.json': '214198c56e4b3668f1de4e535a35130828097052bf3914e6eef5c14260aa1d73', 'artifacts/phase1/environment.json': '0c297add6404b6a1035b71c2cec0737c3af9b6b2ebd8a82f8e8cc81844ed688b', 'artifacts/phase4/litecrack_ba/checkpoints/best.json': '6a35e1ae9fac9cd31ca5c4ed2cc03e29c3bc5dd87f89d7658374446963e5521c', 'artifacts/phase4/litecrack_ba/checkpoints/latest.json': 'a5f474701d38d5444062835fbcff0585b0605b5a3cacbd7767026915d515ddbf', 'artifacts/phase4/litecrack_ba/checkpoints/step-000002580-2f498b1b.pt': '786fbf654af8ff1b0e139328ac5d4f121ce9c215dd504988054ae21502f661f6', 'artifacts/phase4/litecrack_ba/checkpoints/step-000003000-4960caeb.pt': '73b25f40d5a1b95aff9eb96408a77478bb0db0396018e699c6cba08aa722e97d', 'artifacts/phase4/litecrack_ba/config.json': '2c24d7ed285069fff948f4499ddc7b531446b195f09711c6230a2955d96dab4a', 'artifacts/phase4/litecrack_ba/environment.json': 'af0cbfe18d91b9170aec83ea3ab636bdb11fbf66c38c74618f1ee4e1efacbd67', 'artifacts/phase4/litecrack_ba/evaluation.json': '4544c76596315cb178196983d22f17da44f28a5f50e491551e71e31a52d314f3', 'artifacts/phase4/litecrack_ba/history.json': '42c1aa8271c087d968e1825151131f4961283d0f5bf4751684010b0e41fefd8a', 'artifacts/phase4/litecrack_ba/manifest.json': '9c97554166af5494c746d46885425f748ea46c4035863c22c90215b3506358ae', 'artifacts/phase4/litecrack_ba/profile_cpu.json': '2f24ed6211b964d86bd552a1b2e57e7090077813ac7c0c773a4243f796bfb0bc', 'artifacts/phase4/litecrack_ba/profile_cuda.json': '84eeb12040ab36eb25da5201dc4b34d56d7eb760e5562c56f651a99778067f04', 'artifacts/phase4/litecrack_ba/source_validation_per_image.json': '4414c9e1f9d3ec79974cd24d7b103df8f614b4c0c07205c6e58e88981aab7a4f', 'artifacts/phase4/litecrack_ba/status.json': '8f8aefb5f615cec836f476d0c28502d3c9c055d5e09dae439bfbc535289f0fc9', 'artifacts/phase4/profile_cpu.log': 'b66107ed8a0ac39905f715d235ffbb3d07e9a2c22b61a9a89210ca8696320374', 'artifacts/phase4/profile_cuda.log': 'f303238aeaf4a17a8283da17427c74c1ff761be30e4d8d30c7e3c59075e4ddba', 'artifacts/phase4/requirements.kaggle.lock.txt': '75abf5a5b84e04e2e3ba62cbad028024968b3336d10544c8479fe9e5f0918313', 'artifacts/phase4/tests.txt': '0d49ffcd662d280476d77cffaf1d662a7fc11ba1b95b93be3630e4298c5d8693', 'artifacts/phase4/tests.xml': '5934cf73729b35853b1be261e9e49267dfee4e0c588f03da9fc071363ab4145b', 'artifacts/phase4/training.log': 'b71648d9a04214f18105d27b775a307346a50029d61be3db026df336edbb35cf', 'configs/baseline_r2au.json': '76f058a412db7f81d4b925223f3e91b53ae5d1b893dda58f65974bb4151d0577', 'configs/baseline_unet.json': '205b44ada0cf895862f3bb31a33041b786b1eed24c880c38b9c083f475c9659c', 'configs/data_sources.json': 'e33ad521df49827fc30af7308541af4fb2a2b69ccd34ac919c19d2ddc5861c1f', 'configs/litecrack.json': 'fc99fafbcd4e4e5c5cf50d57d027271e71b0b0f4830af7196ab896a20c925fa4', 'configs/litecrack_ba.json': '413c7b14d74e0275bfc1df2abe0b3b13c7423262a3b11607d3da6f84ccff526e', 'configs/phase1.json': '51484e20e224486e7db55fab2ed8a0ded307cf2962fd53467c4b5311c1caf1e4', 'configs/stabilized_r2au.json': '197ffd9b58eec40d9e3e5de225e6890d869424e5900bbe2471dd15535a020be3', 'configs/stabilized_unet.json': 'fcd8006ecbd6c4ecbfcbb33c850c7736868a443dd95aa5551a3edc8bf2d03e08', 'data/manifests/baseline_registry_v1.json': '365e9d8c74d39e6bebf6f82c980a0148f5708bbfc17886adc688edca9e8c1f21', 'data/manifests/pilot_v1.json': '9c97554166af5494c746d46885425f748ea46c4035863c22c90215b3506358ae', 'data/manifests/pilot_v1.json.sha256': 'e04269dae5566234d83b5c20a6c631baaba57092fbb4e62dcf0f910c3bb05364', 'requirements.txt': '6f109fbff5a330a6e0f57b1a687e466ab0b330fa3921774e5e0310e65e3ee512', 'scripts/audit_kaggle_data.py': '463dce488a03d7a432df6c16bba74ae619fbf41b08b413e27aa90e6e2fa222da', 'scripts/build_data_audit_notebook.py': '68c90fa0accc72b22022ed11994d038f1812a7cb19f295d84ccadf50d1029ba0', 'scripts/build_final_evaluation_notebook.py': 'caf2d85b5d6d1b57eacd4f43bcd7116382e84a169c99acfa3013d8975b4098f2', 'scripts/build_kaggle_notebook.py': 'd121b7956103c42e9f2b340387286f623172e8e01792ddcaa358541a468af79d', 'scripts/build_phase2_notebook.py': 'dc05c84ebad18b9f3828f78563bb75bf2d954eea3874d428b68eb36e9390d8e0', 'scripts/build_phase3_evaluation_notebook.py': '8b96ece4f2a1c0d7196f9b6a5e38b3324e093ce5dc85908808de9cd06b4493af', 'scripts/environment.py': 'e0386ae0d9a9ef13a5fb27d3464b2afa2faa1181a055fbaf3b9aae48702183c3', 'scripts/evaluate_frozen_baselines.py': '864af8feb41d08786629a4f0f2ef4aea8f7c69e365e60ef8fcc3830890f1f6b8', 'scripts/evaluate_litecrack.py': 'de8888c6479547b7436ea25955242bb5f641c051c9f82cc2132e245008fd9a69', 'scripts/freeze_pilot.py': 'bf7edba7258f0ab9750ff1197b12cce850f71b0d278de3fa353399b9ad45de85', 'scripts/prepare_data.py': 'fc59ddd85cde50d1e30d7d988ad8c2f898c11deb0223119e66149a67c3e54c25', 'scripts/profile_baseline.py': 'f62e6f9e121255193add4ef575d5c77d2503b02337dcacac4e483cdc445faa41', 'scripts/restore_datasets.py': '0d7fc8fd923c817c1da393abe2b76a4b49625a433f8724a88b62bd97f2791a5b', 'scripts/train_baseline.py': '1cb6fab153d411da701a1ebf8fdd4bbd8eaa48f16617c938e108b08d816143cf', 'src/__init__.py': '8ccf2c63120e3ea084ae3827cd5d4189d092f266d9ce6e11a97fb2b5aa175bef', 'src/atomic.py': '8d3e04bad91ed969540d145613c83b719c69f3346bf5ced3bda46dec5be84823', 'src/boundaries.py': '87d542db7182a5ea220c760710c82948949ddad662ecec2d4876dff41564ee0b', 'src/checkpoints.py': '0a34e05d17f4b4d255abdc48b38b6c429a6427ac8fe351d9a1204d40bd783264', 'src/dataset.py': 'a472d7c679edd5fadb974b8ef168db29ddcf2288a2beca23345a0e3379fafc57', 'src/litecrack.py': '9f0fce2b4bb1aa5557fb763ac93a65a23a80a962a3d4294df6a1d8116e2339b0', 'src/model_factory.py': '18a12ebb15276f51fe807988f56ae7caefa66a3b0cc485f146a9da7475c10fce', 'src/models.py': '011980f40fc4c97b2a42a39048231d10e157e5e38bb70ffd6ef33f27de1214e5', 'src/segmentation.py': '9016f5ba4cb859c9f448e3889b0d8f04cd3a3f5848e5ca59898db96cbdb082b5', 'tests/test_boundaries.py': '8d6f1607773c424d92a20a7aa59bae9228e958fc8a2074af6033fbedf39eeef7', 'tests/test_checkpoint_resume.py': 'fc852c67163aaee31d4cb93b64b4c077fc888d52ad5eb99778553c3287f0a56e', 'tests/test_data_audit.py': '222b4de7fb5c2efed6183b95c90d8b0ca173c3e59c1ef19d9a4362cd3ab0d277', 'tests/test_dataset_restore.py': 'f480b83820f37077d605546b692483af8dbf844b3347a494bd7c28fe6f4ac7c4', 'tests/test_foundation.py': 'b511f320a296782b354c4169d4fff15428180dbc436aede61aa52b2c2c778c39', 'tests/test_litecrack.py': '9d2d3021527adbbf8a60a1a5ba9bc43f5b0b60fb3933342db41ec397b8fcb3e3', 'tests/test_pair_discovery.py': '78873635f09924fcd8f0174dda4cf8621576f22499807ab60b665a3f74cff440', 'tests/test_phase2.py': '49f7cdf660a74b4087b92e8f5528c37624fc30cdf64b83b8e4ace1936c7817e9'}}]
import os,sys,json,hashlib,zipfile,shutil,subprocess
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
project=Path('/kaggle/working/road_crack_boundary_comparison_v1')
project.mkdir(exist_ok=True)
for name,text in files.items():
    path=project/name; path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists(): assert path.read_text()==text, 'Existing comparison code differs; preserve it'
    else: path.write_text(text)
projects={}
for spec in specs:
    phase=spec['phase']; name='litecrack' if phase==3 else 'litecrack_ba'
    destination=Path(f'/kaggle/working/road_crack_phase{phase}_v1')
    marker=f'artifacts/phase{phase}/{name}/checkpoints/best.json'
    if not (destination/marker).exists():
        archives=list(Path('/kaggle/input').rglob(spec['archive']))
        local=Path('/kaggle/working')/spec['archive']
        if local.exists(): archives.insert(0,local)
        destination.mkdir(exist_ok=True)
        if archives:
            assert hashlib.sha256(archives[0].read_bytes()).hexdigest()==spec['sha256'], 'Unexpected backup'
            with zipfile.ZipFile(archives[0]) as z:
                for member in z.namelist():
                    target=(destination/member).resolve(); assert target.is_relative_to(destination.resolve())
                    if target.exists(): assert target.read_bytes()==z.read(member), 'Existing file differs'
                z.extractall(destination)
        else:
            roots=[p.parents[4] for p in Path('/kaggle/input').rglob('best.json') if p.as_posix().endswith('/'+marker)]
            valid=[r for r in roots if all((r/n).is_file() and hashlib.sha256((r/n).read_bytes()).hexdigest()==h for n,h in spec['hashes'].items())]
            assert valid, 'Attach '+spec['archive']+' or its complete extracted contents'
            for n,h in spec['hashes'].items():
                target=destination/n
                if target.exists(): assert hashlib.sha256(target.read_bytes()).hexdigest()==h, 'Existing file differs'
            for n in spec['hashes']:
                target=destination/n; target.parent.mkdir(parents=True,exist_ok=True)
                if not target.exists(): shutil.copyfile(valid[0]/n,target)
    projects[phase]=destination
os.chdir(project)
restore_report=project/'dataset_restore.json'
subprocess.run([sys.executable,'scripts/restore_datasets.py','--manifest',str(projects[4]/'data/manifests/pilot_v1.json'),
    '--sources','configs/data_sources.json','--destination','/kaggle/working/road_crack_phase1/data/raw','--report',str(restore_report)],check=True)
data_root=Path(json.loads(restore_report.read_text())['data_root'])


Reused existing data; all image/mask checksums passed: /kaggle/working/road_crack_phase1/data/raw


In [12]:
from datetime import datetime,timezone
output=project/('results_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
result=subprocess.run([sys.executable,'scripts/compare_boundaries.py','--control',str(projects[3]),
    '--treatment',str(projects[4]),'--data-root',str(data_root),'--output',str(output)],capture_output=True,text=True)
log=output.with_suffix('.log'); log.write_text(result.stdout+result.stderr)
print(result.stdout,result.stderr)
assert result.returncode==0, 'Comparison failed; preserve the log for diagnosis'


{
  "manifest_sha256": "9c97554166af5494c746d46885425f748ea46c4035863c22c90215b3506358ae",
  "checkpoints": {
    "control": {
      "checkpoint_sha256": "9ac52e68c86938f826ddf9bf5176ce8abba830c4ecb3cdd01dd35823777c1fa2",
      "best_epoch": 99,
      "validation_replay": {
        "loss": 0.40793500741322836,
        "dice": 0.7372418018368383,
        "iou": 0.6223634694780533,
        "precision": 0.7131622011773289,
        "recall": 0.7819713837156693,
        "boundary_f1": 0.8579921356712779,
        "predicted_foreground_fraction": 0.02988154093424479,
        "truth_foreground_fraction": 0.028463999430338543,
        "n": 60
      }
    },
    "boundary": {
      "checkpoint_sha256": "786fbf654af8ff1b0e139328ac5d4f121ce9c215dd504988054ae21502f661f6",
      "best_epoch": 86,
      "validation_replay": {
        "loss": 0.5334622581799825,
        "dice": 0.7316660778907438,
        "iou": 0.6144075044469598,
        "precision": 0.7213073857477866,
        "recall": 0.782301175

In [13]:
backup=Path('/kaggle/working/phase4_comparison_results.zip')
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(output.rglob('*')):
        if path.is_file(): z.write(path,path.relative_to(output).as_posix())
    for name in ['scripts/compare_boundaries.py','src/boundaries.py']:
        z.write(project/name,name)
    z.write(log,'execution.log')
digest=hashlib.sha256(backup.read_bytes()).hexdigest()
backup.with_suffix('.zip.sha256').write_text(digest+'  '+backup.name+'\n')
print('Download from Kaggle Files:',backup)
print('SHA256:',digest)


Download from Kaggle Files: /kaggle/working/phase4_comparison_results.zip
SHA256: c145f89d6b7174ed409f6864b5e6d3d5744fc89a7872e5ee4dd8fce369fe91fe
